
# 04 - Gold Layer: Data Aggregation
# Create Business Metrics and Analytical Tables


In [0]:
# Imports
import uuid
import traceback

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from pyspark.sql import Row
from pyspark.sql.types import *

In [0]:
# Configuración
pipeline_name = "mlb_daily_games_etl"
timezone = "America/Santo_Domingo"

pipeline_run_id = str(uuid.uuid4())
current_time_rd = datetime.now(ZoneInfo(timezone))

execution_date = (current_time_rd - timedelta(days=1)).strftime("%Y-%m-%d")

print(f"Pipeline Run ID: {pipeline_run_id}")
print(f"Execution Date: {execution_date}")

In [0]:
# Función de logging corregida
def log_event(layer, status, message, records_processed, started_at):
    ended_at = datetime.now(ZoneInfo(timezone))
    duration_seconds = float((ended_at - started_at).total_seconds())

    log_schema = StructType([
        StructField("pipeline_name", StringType(), True),
        StructField("pipeline_run_id", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("status", StringType(), True),
        StructField("message", StringType(), True),
        StructField("execution_date", StringType(), True),
        StructField("records_processed", IntegerType(), True),
        StructField("started_at", TimestampType(), True),
        StructField("ended_at", TimestampType(), True),
        StructField("duration_seconds", DoubleType(), True)
    ])

    log_row = [(
        pipeline_name,
        pipeline_run_id,
        layer,
        status,
        message,
        execution_date,
        int(records_processed),
        started_at,
        ended_at,
        duration_seconds
    )]

    log_df = spark.createDataFrame(log_row, schema=log_schema)

    log_df.write.format("delta").mode("append").saveAsTable(
        "mlb_project.pipeline_logs"
    )

In [0]:
# Validar que existan datos en Silver
gold_started_at = datetime.now(ZoneInfo(timezone))

try:
    silver_count = spark.sql(f"""
    SELECT COUNT(*) AS total_records
    FROM mlb_project.silver_mlb_games
    WHERE game_date = '{execution_date}'
    """).collect()[0]["total_records"]

    if silver_count == 0:
        raise Exception(f"No Silver data found for date: {execution_date}")

    print(f"Silver records found: {silver_count}")

except Exception:
    error_message = traceback.format_exc()

    log_event(
        layer="gold",
        status="FAILED",
        message=error_message,
        records_processed=0,
        started_at=gold_started_at
    )

    raise Exception(error_message)

In [0]:
# Crear DataFrame Gold
try:
    gold_df = spark.sql(f"""
    SELECT
        game_date,
        COUNT(*) AS total_games,
        SUM(CASE WHEN is_final = true THEN 1 ELSE 0 END) AS final_games,
        SUM(CASE WHEN is_final = false THEN 1 ELSE 0 END) AS non_final_games,
        SUM(COALESCE(home_score, 0) + COALESCE(away_score, 0)) AS total_runs,
        ROUND(AVG(COALESCE(home_score, 0) + COALESCE(away_score, 0)), 2) AS avg_runs_per_game,
        COUNT(DISTINCT home_team_id) + COUNT(DISTINCT away_team_id) AS teams_playing,
        CURRENT_TIMESTAMP() AS last_updated_at,
        '{pipeline_run_id}' AS pipeline_run_id
    FROM mlb_project.silver_mlb_games
    WHERE game_date = '{execution_date}'
    GROUP BY game_date
    """)

    print("Gold DataFrame created successfully.")
    display(gold_df)

except Exception:
    error_message = traceback.format_exc()

    log_event(
        layer="gold",
        status="FAILED",
        message=error_message,
        records_processed=0,
        started_at=gold_started_at
    )

    raise Exception(error_message)

In [0]:
# MERGE hacia Gold
try:
    gold_df.createOrReplaceTempView("gold_updates")

    spark.sql("""
    MERGE INTO mlb_project.gold_mlb_daily_summary AS target
    USING gold_updates AS source
    ON target.game_date = source.game_date

    WHEN MATCHED THEN UPDATE SET
        target.total_games = source.total_games,
        target.final_games = source.final_games,
        target.non_final_games = source.non_final_games,
        target.total_runs = source.total_runs,
        target.avg_runs_per_game = source.avg_runs_per_game,
        target.teams_playing = source.teams_playing,
        target.last_updated_at = source.last_updated_at,
        target.pipeline_run_id = source.pipeline_run_id

    WHEN NOT MATCHED THEN INSERT *
    """)

    log_event(
        layer="gold",
        status="SUCCESS",
        message="Gold MERGE completed successfully.",
        records_processed=gold_df.count(),
        started_at=gold_started_at
    )

    print("Gold MERGE completed successfully.")

except Exception:
    error_message = traceback.format_exc()

    log_event(
        layer="gold",
        status="FAILED",
        message=error_message,
        records_processed=0,
        started_at=gold_started_at
    )

    raise Exception(error_message)

In [0]:
# Validar Gold
display(
    spark.sql("""
    SELECT *
    FROM mlb_project.gold_mlb_daily_summary
    ORDER BY game_date DESC
    LIMIT 20
    """)
)

In [0]:
# Revisar logs del pipeline
display(
    spark.sql("""
    SELECT *
    FROM mlb_project.pipeline_logs
    ORDER BY started_at DESC
    LIMIT 20
    """)
)